# M2 균형학습 체크포인트 진단

완료된 seed 42 M1·균형학습 M2 체크포인트만 읽습니다. **재학습과 모델 선택은 하지 않습니다.**

출력: (1) ID/N/V 블록별 성과, (2) ID-only 대비 축별 증분, (3) CLV 구간별 가격·구매금액 가중 정답상품 순위이동, (4) 실제 후보점수의 ID/N/V 영향력.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REVIEWED_SHA = 'b322f59c1227119aa18a4a6dce7a301c91459e92'
!rm -rf /content/clv-m2-lightgcn-runner
!git clone -q https://github.com/jung-un/clv-m2-lightgcn-runner.git /content/clv-m2-lightgcn-runner
%cd /content/clv-m2-lightgcn-runner
!git checkout -q {REVIEWED_SHA}

import subprocess
assert subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip() == REVIEWED_SHA
print('진단 코드 SHA 확인:', REVIEWED_SHA)

In [ ]:
import json
from lightgcn_clv_gatefree_lowdim_balanced_diagnostic import (
    configure_balanced_checkpoint_diagnostic,
    preflight_summary,
    run_balanced_checkpoint_diagnostic,
)

cfg = configure_balanced_checkpoint_diagnostic(
    out_dir=(
        '/content/drive/MyDrive/논문/data/'
        'results_v3_dunnhumby_m2_gatefree_lowdim_balanced_training_historical_screen_v1'
    ),
    baseline_result_dir=(
        '/content/drive/MyDrive/논문/data/'
        'results_v3_dunnhumby_m2_repeatshare_historical_backtest_v1'
    ),
    eval_batch_size=32,
)
print(json.dumps(preflight_summary(cfg), ensure_ascii=False, indent=2))

In [ ]:
# optimizer·epoch 학습 없이 기존 checkpoint를 재평가합니다.
report = run_balanced_checkpoint_diagnostic(cfg)

In [ ]:
from IPython.display import display

print('1) ID/N/V 블록별 성과')
display(report['view_metrics'])

print('2) 같이 학습된 ID-only 대비 N축·V축·두 축 증분')
display(report['axis_attribution'])

print('3) CLV 구간별 가격·구매금액 가중 정답상품 순위이동 요약')
display(report['movement_summary'])

print('4) 순위 구간별 상세 이동')
display(report['rank_transition'])

print('5) 실제 후보점수 영향력')
display(report['score_strength'])

print('저장 파일')
print(json.dumps(report['paths'], ensure_ascii=False, indent=2))